In [1]:
# 1. Install dependencies
!pip install dm_control

# 2. Clone TD-MPC into the tdmpc/ folder inside your project
%cd /content
!git clone https://github.com/odykon/td-mpc_o2_local

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.7 MB/s eta 0:00:00
/content
Cloning into 'td-mpc_o2_local'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 82 (delta 14), reused 23 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.50 MiB | 14.76 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [2]:
import sys
sys.path.insert(0, '/content/td-mpc_o2_local')
sys.path.insert(0, '/content/td-mpc_o2_local/tdmpc/src')

In [45]:
from pathlib import Path
from cfg import parse_cfg
from omegaconf import OmegaConf
from env import make_env

def make_cfg(task, cfg_path=Path('/content/td-mpc_o2_local/tdmpc/cfgs')):
    import sys
    # clear any previous task argument
    sys.argv = [arg for arg in sys.argv if not arg.startswith('task=')]
    sys.argv += [f'task={task}']
    return parse_cfg(cfg_path)

# load the base config
cfg = make_cfg('humanoid-stand')


for key, value in cfg.items():
    print(f"{key}: {value}")



task: humanoid-stand
modality: state
action_repeat: 2
discount: 0.99
episode_length: 500
train_steps: 1500000
iterations: 12
num_samples: 512
num_elites: 64
mixture_coef: 0.05
min_std: 0.05
temperature: 0.5
momentum: 0.1
batch_size: 512
max_buffer_size: 1000000
horizon: 5
reward_coef: 0.5
value_coef: 0.1
consistency_coef: 2
rho: 0.5
kappa: 0.1
lr: 0.001
std_schedule: linear(0.5, 0.05, 25000)
horizon_schedule: linear(1, 5, 25000)
per_alpha: 0.6
per_beta: 0.4
grad_clip_norm: 10
seed_steps: 5000
update_freq: 2
tau: 0.01
enc_dim: 256
mlp_dim: 512
latent_dim: 100
use_wandb: False
wandb_project: none
wandb_entity: none
seed: 1
exp_name: default
eval_freq: 20000
eval_episodes: 10
save_video: False
save_model: False
-f: True
/root/: {'local/share/jupyter/runtime/kernel-118db7ce-8bd0-4ede-817d-64fc9146edd9': {'json': None}}
task_title: Humanoid Stand
device: cuda


In [48]:
cfg = OmegaConf.merge(cfg, OmegaConf.create({
    """
    # planning
    'num_samples':  512,
    'num_elites':   128,
    'iterations':   12,
    'momentum':     0.1,
    """
    # decoder
    'decoder_pretraining': False,
    'decoder_init':        True,
    'latent_action_dim':   128,
    'use_latent_state':    True,
    'dcem_batch_size':     64,
    'decoder_updates':     70,
    'dcem_sampling_n':     20000,
    'lml_temperature':     10.0,

    # TOLD learning
    'batch_size':    512,
    'told_updates':  500,

    # evaluation
    'eval_episodes': 0,
    'video_mode':    'none',

    #training
    'train_steps': 4000,
    'seed_steps': 1000,
    'horizon_schedule': 'linear(1, ${horizon}, 4000)',
    'std_schedule': 'linear(0.5, ${min_std}, 4000)',
    #scheduling

}))

In [50]:
from implementation import DCEM_TDMPC
from algorithm.helper import linear_schedule
from algorithm.helper import Episode, ReplayBuffer
import implementation.logging as l
import time
import torch

In [49]:
env = make_env(cfg)
agent = DCEM_TDMPC(cfg)
buffer = ReplayBuffer(cfg)

In [51]:
RESULTS_DIR = '/content/results'

In [52]:
episode_idx, start_time = 0, time.time()
print("\n" + "=" * 50)
print("\033[1m🚀 Training Configuration\033[0m")
print("=" * 50)
for key, value in agent.cfg.items():
    print(f"{key:25}: {value}")
print("=" * 50 + "\n")
save_dir = l.make_save_dir_path(agent.cfg, base_dir = RESULTS_DIR)
print("Saving results to:", save_dir)

all_metrics = []

for step in range(0, cfg.train_steps, cfg.episode_length):

    obs =env.reset()
    episode = Episode(cfg, obs)
    half_time_reward= 0

    episode_start_time = time.time()
    while not episode.done:

        if(step<cfg.seed_steps):
            random_action_np = env.action_space.sample() # Choose a random action (NumPy array)
            action = torch.tensor(random_action_np, dtype=torch.float32, device=agent.device)

        else:
            #action = agent.plan(obs, eval_mode =False,step=step, t0=episode.first)
            action, _, _, _, _ = agent.CEM_in_latent(obs, step = step, t0=episode.first, seed=None, sample_final_action =True)
        obs, reward, done, _ = env.step(action.cpu().numpy())

        if(episode._idx<=500):
            half_time_reward+=reward

        episode+= (obs,action,reward,done)
    episode_end_time = time.time()
    horizon = int(linear_schedule(cfg.horizon_schedule,step))
    episode_metrics = {
        'Episode_no:': int(step/cfg.episode_length),
        'Reward': episode.cumulative_reward,
        'Half_time_reward': half_time_reward,
        'Horizon': int(linear_schedule(cfg.horizon_schedule,step)),
        'Std:': linear_schedule(cfg.std_schedule,step)
    }
    buffer+=episode
    print("Buffer idx:", buffer.idx,", buffer is full:", buffer._full)

    print("\n  Episode Summary")
    print("-" * 25)
    for k, v in episode_metrics.items():
        print(f"  {k:15}: {v}")


    train_metrics = {}

    if (step>= cfg.seed_steps):
        model_update_start_time = time.time()
        metrics = update_tdmpc(agent, buffer, step)
        model_update_end_time = time.time()


    """
    if (step>= cfg.seed_steps):
        decoder_update_start_time = time.time()
        decoder_loss = update_decoder(agent, buffer, cfg, step)
        train_metrics['Decoder_loss'] = decoder_loss
        decoder_update_end_time = time.time()
    """

    print("  Training Metrics:")
    for k, v in metrics.items():
        print(f"  {k:15}: {v}")



    """
    if (step%4000 == 0 and step>=cfg.seed_steps and cfg.eval_episodes>0):
        full_evaluation_metrics = l.evaluate_agent(env, agent, agent.cfg, step, cem=False, n_episodes = cfg.eval_episodes, save_dir = save_dir,video_mode=cfg.video_mode)
        eval_summary = {
            "step": eval_metrics.get("step"),
            "mean_reward": eval_metrics.get("mean_reward"),
            "std_reward": eval_metrics.get("std_reward"),
        }
    else: eval_summary = None

    if(step>=cfg.seed_steps):
        l.save_results(agent.cfg, full_metrics, save_dir, eval_summary, step)
    all_metrics.append(full_metrics)

#Evaluate the final agent
final_evaluation_metrics = l.evaluate_agent(env, agent, agent.cfg, step, cem=False, n_episodes = 2, save_dir = save_dir,video_mode='none')
path = os.path.join(save_dir, "final_eval.csv")
df = pd.DataFrame([final_evaluation_metrics])
if os.path.exists(path):
    df.to_csv(path, mode="a", header=False, index=False)
else:
    df.to_csv(path, index=False)
l.save_model_and_buffer(agent, buffer, save_dir)
"""



🚀 Training Configuration
task                     : humanoid-stand
modality                 : state
action_repeat            : 2
discount                 : 0.99
episode_length           : 500
train_steps              : 4000
iterations               : 12
num_samples              : 512
num_elites               : 64
mixture_coef             : 0.05
min_std                  : 0.05
temperature              : 0.5
momentum                 : 0.1
batch_size               : 512
max_buffer_size          : 1000000
horizon                  : 5
reward_coef              : 0.5
value_coef               : 0.1
consistency_coef         : 2
rho                      : 0.5
kappa                    : 0.1
lr                       : 0.001
std_schedule             : linear(0.5, 0.05, 4000)
horizon_schedule         : linear(1, 5, 4000)
per_alpha                : 0.6
per_beta                 : 0.4
grad_clip_norm           : 10
seed_steps               : 1000
update_freq              : 2
tau                      : 

KeyboardInterrupt: 

In [20]:
def update_tdmpc(agent, buffer, step):
    """Original TD-MPC update — untouched."""
    agent.model.track_TOLD_grad(True)
    buffer.cfg.batch_size = agent.cfg.batch_size
    metrics = {}
    for i in range(agent.cfg.told_updates):
        update_metrics = agent.update(buffer, step + i)
        for k, v in update_metrics.items():
            metrics[k] = metrics.get(k, 0.0) + v
    for k in metrics:
        metrics[k] /= agent.cfg.told_updates
    return metrics

def update_decoder(agent, buffer, cfg, step):
    """DCEM decoder update."""
    agent.model.track_TOLD_grad(False)
    buffer.cfg.batch_size = agent.cfg.dcem_batch_size
    horizon = int(linear_schedule(cfg.horizon_schedule, step))
    decoder_loss = 0
    for i in range(agent.cfg.decoder_updates):
        obs, *_ = buffer.sample()
        _, u_mean, _, _, _ = agent.DCEMethod(obs, update_mode=True, step=step, t0=False)
        decoder_loss += agent.action_decoder_DDPG_update(obs, u_mean, horizon)
    agent.model.track_TOLD_grad(True)
    return decoder_loss / agent.cfg.decoder_updates

def collect_episode(env, agent, cfg, step):
    """Collect one episode of experience."""
    obs = env.reset()
    episode = Episode(cfg, obs)
    while not episode.done:
        if step < cfg.seed_steps:
            action_np = env.action_space.sample()
            action = torch.tensor(action_np, dtype=torch.float32, device=agent.device)
        else:
            action, _, _, _, _ = agent.CEM_in_latent(
                obs, step=step, t0=episode.first, seed=None, sample_final_action=True
            )
        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    return episode


/content/td-mpc_o2_local


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [53]:
%cd /content/td-mpc_o2_local
!git status
"""
!git config user.email "odysseaskon@gmail.com"
!git config user.name "odykon"
!git add .
!git commit -m "updated files for training"
!git push https://@github.com/odykon/td-mpc_o2_local.git main
"""

/content/td-mpc_o2_local
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


'\n!git config user.email "odysseaskon@gmail.com"\n!git config user.name "odykon"\n!git add .\n!git commit -m "updated files for training"\n!git push https://@github.com/odykon/td-mpc_o2_local.git main\n'

In [22]:
# search for all uses of action_repeat in the codebase
import subprocess
result = subprocess.run(
    ['grep', '-rn', 'action_repeat', '/content/td-mpc_o2_local/tdmpc'],
    capture_output=True, text=True
)
print(result.stdout)

/content/td-mpc_o2_local/tdmpc/src/logger.py:33:		   ('train steps', f'{int(cfg.train_steps*cfg.action_repeat):,}'),
/content/td-mpc_o2_local/tdmpc/src/env.py:174:	def __init__(self, env, domain, task, action_repeat, modality):
/content/td-mpc_o2_local/tdmpc/src/env.py:210:		self.ep_len = 1000//action_repeat
/content/td-mpc_o2_local/tdmpc/src/env.py:267:	env = ActionRepeatWrapper(env, cfg.action_repeat)
/content/td-mpc_o2_local/tdmpc/src/env.py:279:	env = TimeStepToGymWrapper(env, domain, task, cfg.action_repeat, cfg.modality)
/content/td-mpc_o2_local/tdmpc/src/train.py:77:		env_step = int(step*cfg.action_repeat)
/content/td-mpc_o2_local/tdmpc/cfgs/pixels.yaml:1:train_steps: 3000000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:4:action_repeat: ???
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:6:episode_length: 1000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/default.yaml:7:train_steps: 500000/${action_repeat}
/content/td-mpc_o2_local/tdmpc/cfgs/tasks/do